# Lab 5 — Environment and Resource Check

**Core duration: 10 minutes. Install packages and obtain weights before class.**

This notebook checks readiness without opening your camera or starting model training. Run it inside `lab05/` or at the repository root.

**Today's route:** environment 10 min → PyTorch 35 min → detection/evaluation 30 min → custom workflow 65 min → team planning 20 min → troubleshooting/discussion 20 min = **180 min**.

Training duration depends on hardware.


## 1. Packages and interpreter (3 min)

Use the same course `.venv`. Run these commands in a terminal **from the repository root**, before class:

Windows:
```text
.venv\Scripts\python.exe -m pip install -r lab05/requirements-lab05.txt
.venv\Scripts\python.exe -m jupyter lab
```
Linux:
```text
.venv/bin/python -m pip install -r lab05/requirements-lab05.txt
.venv/bin/python -m jupyter lab
```
Use the [official PyTorch installer](https://pytorch.org/get-started/locally/) if your hardware needs a specific build. Restart the notebook kernel after installation. The digit network uses CPU; detector training may take longer.


In [ ]:
from pathlib import Path
# Support notebooks started in lab05/ or in the repository root.
LAB_DIR = Path.cwd().resolve()
if (LAB_DIR / "lab05" / "data" / "digits.csv").is_file():
    LAB_DIR = LAB_DIR / "lab05"
if not (LAB_DIR / "data" / "digits.csv").is_file():
    raise FileNotFoundError("Run this notebook in lab05/ or the repository root; keep data/ and scripts/ in place.")

import sys, importlib
print("Python:", sys.version.split()[0])
print("Interpreter:", sys.executable)
print("Lab directory:", LAB_DIR)
checks = {"Python >= 3.11": sys.version_info >= (3, 11)}
modules = {}
for name in ["numpy", "pandas", "matplotlib", "sklearn", "torch", "ultralytics", "cv2", "PIL", "yaml"]:
    try:
        modules[name] = importlib.import_module(name)
        checks["Import " + name] = True
        print(name, getattr(modules[name], "__version__", "OK"))
    except Exception as exc:
        checks["Import " + name] = False
        print(name, "NOT READY:", type(exc).__name__, str(exc))


## 2. Local files and model weights (3 min)

The sample photo/video are included, but YOLO weights are **not** in the supplied repo. 

Set `ALLOW_DOWNLOAD=True` in this cell only if you want Ultralytics to download the weights now. The same setting is available independently in Notebooks 02 and 03. When you have downloaded the weights once, it should be available for following labs.


In [ ]:
for name in ["data/digits.csv", "data/sample.jpg", "data/sample_video.mp4",
             "examples/yolo_dataset/data.yaml", "scripts/webcam_detection.py", "scripts/webcam_capture.py"]:
    checks["File " + name] = (LAB_DIR / name).is_file()
# Set this once before class. Download only when explicitly enabled.
ALLOW_DOWNLOAD = False
weight_candidates = [LAB_DIR / "weights/yolo11n.pt", LAB_DIR / "yolo11n.pt", LAB_DIR.parent / "yolo11n.pt"]
WEIGHTS = next((p for p in weight_candidates if p.is_file()), weight_candidates[0])
MODEL_SOURCE = str(WEIGHTS) if WEIGHTS.is_file() else ("yolo11n.pt" if ALLOW_DOWNLOAD else None)

if ALLOW_DOWNLOAD and not WEIGHTS.is_file() and "ultralytics" in modules:
    modules["ultralytics"].YOLO("yolo11n.pt")
    downloaded = Path("yolo11n.pt").resolve()
    if downloaded.is_file():
        WEIGHTS = downloaded
checks["Local YOLO weights"] = WEIGHTS.is_file()
print("Weights:", WEIGHTS, "available" if WEIGHTS.is_file() else "MISSING")


## 3. Functional checks (4 min)

Check a tensor, an image, a video frame, and (if available) one local detector prediction. A successful package import alone does not establish functional readiness. Camera permissions and desktop windows are checked later using the provided scripts.


In [ ]:
checks["PyTorch tensor"] = False
if "torch" in modules:
    try:
        torch = modules["torch"]
        checks["PyTorch tensor"] = abs(torch.tensor([1., 2., 3.]).mean().item() - 2.) < 1e-6
        print("CUDA available:", torch.cuda.is_available(), "| CPU used by default")
    except Exception as exc:
        print("Tensor test failed:", exc)
checks["Sample image"] = False
if "PIL" in modules:
    try:
        from PIL import Image
        with Image.open(LAB_DIR / "data/sample.jpg") as im:
            im.load()
            checks["Sample image"] = im.width > 0 and im.height > 0
            print("Image size:", im.size)
    except Exception as exc:
        print("Image test failed:", exc)
checks["Video decoding"] = False
if "cv2" in modules:
    cv2 = modules["cv2"]
    cap = cv2.VideoCapture(str(LAB_DIR / "data/sample_video.mp4"))
    try:
        ok, frame = cap.read()
        checks["Video decoding"] = bool(ok and frame is not None)
    finally:
        cap.release()
checks["YOLO inference"] = False
if "ultralytics" in modules and WEIGHTS.is_file():
    try:
        detector = modules["ultralytics"].YOLO(str(WEIGHTS))
        predictions = detector.predict(str(LAB_DIR / "data/sample.jpg"), device="cpu", imgsz=320, verbose=False)
        checks["YOLO inference"] = len(predictions) == 1
        print("Detections:", len(predictions[0].boxes), "(zero detections is not an execution error)")
    except Exception as exc:
        print("Inference test failed:", type(exc).__name__, str(exc))

print("\nREADINESS SUMMARY")
for name, passed in checks.items():
    print("PASS" if passed else "NOT READY", "—", name)
print("READY FOR CORE EXERCISES" if all(checks.values()) else "Resolve NOT READY items before class; skipped checks are not passes.")


### Checkpoint

Show the instructor any failed check and its error message. This notebook verifies inference resources, **not** detector training speed or camera access.

**Next:** `01-pytorch-neural-network.ipynb`.
